# Tự Động Huấn Luyện & So Sánh Nhiều Cấu Hình (Chạy Được Cả Local & Kaggle)
Notebook này được tối ưu sẵn để **chạy trực tiếp trên Kaggle GPU (T4 / P100)** hoặc trên máy cá nhân:
1. **Tự động nhận diện môi trường:** Tự động tìm đường dẫn dataset GoPro.
2. **Tích hợp Dataset:** Khắc phục lỗi thiếu file `utils.dataset` bằng cách khai báo trực tiếp class `GoProDeblurDataset`.
3. **Danh sách cấu hình (`CONFIG_LIST`):** Tự động huấn luyện lần lượt từng cấu hình.
4. **Lưu trữ riêng biệt:** Mỗi cấu hình có file checkpoint `.pth` riêng.
5. **Bảng xếp hạng:** Tự động xuất bảng so sánh PSNR, SSIM, Loss.
6. **Fine-Tuning & Visual Demo:** Tinh chỉnh mô hình tốt nhất và render ảnh so sánh.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import time
import random
import gc
import json
import urllib.request
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Tạo thư mục weights riêng để lưu file .pth cho gọn gàng
os.makedirs("weights", exist_ok=True)

candidate_paths = [
    '.', '..', 
    '/kaggle/working', 
    '/kaggle/working/Image-processing-project-integrating-MoE-and-Q-Former'
]

for p in candidate_paths:
    if os.path.exists(os.path.join(p, 'models')) and os.path.exists(os.path.join(p, 'utils')):
        abs_p = os.path.abspath(p)
        if abs_p not in sys.path:
            sys.path.insert(0, abs_p)
        break

# Bỏ import GoProDeblurDataset ở đây vì ta sẽ định nghĩa lại nó bên dưới để tránh lỗi
try:
    from models.forge import NAFNetBaseline, FORGEDeblurNet
    from utils.metrics import CharbonnierLoss, calc_psnr, calc_ssim, count_parameters
except ImportError as e:
    print(f"Cảnh báo Import: {e}. Vui lòng đảm bảo các thư mục models/ và utils/ tồn tại.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")

pretrained_url = "https://huggingface.co/Jacid23/third-eye-models/resolve/5c2e322323888d8d26852f06715ccc6eb61ea795/NAFNet-GoPro-width64.pth"
nafnet_ckpt_path = os.path.join("weights", "NAFNet-GoPro-width64.pth")

if not os.path.exists(nafnet_ckpt_path):
    print("--> Đang tải pretrained NAFNet từ Hugging Face vào thư mục weights/...")
    urllib.request.urlretrieve(pretrained_url, nafnet_ckpt_path)
    print("--> Tải xuống checkpoint thành công!")
else:
    print(f"--> Đã tìm thấy checkpoint tại: {nafnet_ckpt_path}")

nafnet_state_dict = torch.load(nafnet_ckpt_path, map_location=device)
if 'params' in nafnet_state_dict:
    nafnet_state_dict = nafnet_state_dict['params']
elif 'model_state_dict' in nafnet_state_dict:
    nafnet_state_dict = nafnet_state_dict['model_state_dict']
print("--> Đã nạp thành công nafnet_state_dict vào bộ nhớ!")

Thiết bị đang sử dụng: cuda
--> Đã tìm thấy checkpoint tại: weights\NAFNet-GoPro-width64.pth
--> Đã nạp thành công nafnet_state_dict vào bộ nhớ!


In [2]:
try:
    from models.forge import NAFNetBaseline, FORGEDeblurNet
    from utils.metrics import CharbonnierLoss, calc_psnr, calc_ssim, count_parameters
    from utils.dataset import GoProDeblurDataset, get_gopro_sequence_splits
except ImportError as e:
    print(f"Cảnh báo Import: {e}. Vui lòng đảm bảo các thư mục models/ và utils/ tồn tại.")

### Khởi tạo Dữ liệu và Hàm Đánh Giá

In [3]:
def evaluate_model(model, loader, desc="Validating"):
    model.eval()
    psnr_list, ssim_list = [], []
    pbar_eval = tqdm(loader, desc=f"[{desc}]", leave=False)
    with torch.no_grad():
        for inputs, targets in pbar_eval:
            inputs, targets = inputs.to(device), targets.to(device)
            preds, _ = model(inputs)
            preds = torch.clamp(preds, 0.0, 1.0)
            psnr_list.append(calc_psnr(preds, targets))
            ssim_list.append(calc_ssim(preds, targets))
    avg_psnr = sum(psnr_list) / max(len(psnr_list), 1)
    avg_ssim = sum(ssim_list) / max(len(ssim_list), 1)
    return avg_psnr, avg_ssim

def auto_detect_gopro():
    for prefix in ['/kaggle/input/*', '/kaggle/input', '..', '.']:
        for path in glob.glob(prefix):
            t_train = os.path.join(path, 'gopro-data', 'train') 
            t_test = os.path.join(path, 'gopro-data', 'test')
            if os.path.exists(t_train):
                return t_train, t_test
    return '../gopro-data/train', '../gopro-data/test'

gopro_train_dir, gopro_test_dir = auto_detect_gopro()
print(f"Train dir: {gopro_train_dir}\nTest dir: {gopro_test_dir}")

train_clips, val_clips = get_gopro_sequence_splits(gopro_train_dir, val_clips_count=4)

# 2. Khởi tạo Dataset dựa trên danh sách clip đã chia một cách độc lập
train_sub_ds = GoProDeblurDataset(
    root_dir=gopro_train_dir, 
    clip_names=train_clips, 
    crop_size=192, 
    is_train=True
)

val_sub_ds = GoProDeblurDataset(
    root_dir=gopro_train_dir, 
    clip_names=val_clips, 
    crop_size=None, # Tập Val có thể đánh giá toàn ảnh hoặc crop tùy cấu hình
    is_train=False
)

test_ds = GoProDeblurDataset(
    root_dir=gopro_test_dir, 
    clip_names=None, # Lấy toàn bộ thư mục test
    crop_size=256, 
    is_train=False
)

# 3. Tạo DataLoader
val_loader = DataLoader(val_sub_ds, batch_size=2, shuffle=False, num_workers=1, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=1, pin_memory=True)

print(f"--> Phân chia Dữ liệu theo Clip: Train = {len(train_sub_ds)} ảnh | Val = {len(val_sub_ds)} ảnh | Test = {len(test_ds)} ảnh")

Train dir: ..\gopro-data\train
Test dir: ..\gopro-data\test
--> Phân chia Dữ liệu theo Clip: Train = 1703 ảnh | Val = 400 ảnh | Test = 1111 ảnh


### Đánh giá baseline NAFNet gốc

In [4]:
baseline_model = NAFNetBaseline(
    img_channel=3, 
    width=64, 
    enc_blk_nums=[1, 1, 1, 28], 
    middle_blk_num=1, 
    dec_blk_nums=[1, 1, 1, 1]
).to(device)

baseline_model.load_state_dict(nafnet_state_dict, strict=True)

print("\n===================================================")
print(" ĐANG ĐÁNH GIÁ PURE NAFNet BASELINE TRÊN TẬP TEST...")
print("===================================================")
baseline_psnr, baseline_ssim = evaluate_model(baseline_model, test_loader, desc="Baseline Test")

print(f"\n KẾT QUẢ PURE NAFNET BASELINE:")
print(f"   + PSNR: {baseline_psnr:.2f} dB")
print(f"   + SSIM: {baseline_ssim:.4f}")

del baseline_model
gc.collect()
torch.cuda.empty_cache()


 ĐANG ĐÁNH GIÁ PURE NAFNet BASELINE TRÊN TẬP TEST...


[Baseline Test]:   0%|          | 0/556 [00:04<?, ?it/s]


 KẾT QUẢ PURE NAFNET BASELINE:
   + PSNR: 33.01 dB
   + SSIM: 0.9430


### Bảng Cấu Hình & Vòng Lặp Huấn Luyện Sàng Lọc

In [5]:
# FIX: num_experts=4, r=16, rb=32 tang capacity MoE
# FIX: aux_weight=1e-4 (giam 100x so voi 0.01) tranh catastrophic forgetting
COMMON_CONFIG = {
    'width': 64,
    'enc_blk_nums': [1, 1, 1, 28],
    'middle_blk_num': 1,
    'dec_blk_nums': [1, 1, 1, 1],
    'num_queries': 16,
    'num_experts': 4,       
    'r': 16,                
    'rb': 32,               
    'aux_weight': 1e-4,     
}

SCREENING_EPOCHS = 5

import math
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-7):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self._epoch = 0
    def step(self):
        self._epoch += 1
        e = self._epoch
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            if e <= self.warmup_epochs:
                lr = base_lr * e / self.warmup_epochs
            else:
                progress = (e - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
            pg['lr'] = lr

CONFIG_LIST = [
    {
        'name': 'Exp1_MoE_LR_3e4',
        'lr': 3e-4,
        'batch_size': 2,
        'freeze_backbone': True,
        'warmup_epochs': 1,
        'description': 'Freeze backbone, MoE-LoRA LR=3e-4 + warmup'
    },
    {
        'name': 'Exp2_MoE_LR_1e4',
        'lr': 1e-4,
        'batch_size': 2,
        'freeze_backbone': True,
        'warmup_epochs': 1,
        'description': 'Freeze backbone, MoE-LoRA LR=1e-4 + warmup'
    },
    {
        'name': 'Exp3_MoE_LR_5e4',
        'lr': 5e-4,
        'batch_size': 2,
        'freeze_backbone': True,
        'warmup_epochs': 1,
        'description': 'Freeze backbone, MoE-LoRA LR=5e-4 + warmup'
    }
]

SCREENING_RESULTS = {}
criterion = CharbonnierLoss()

# ĐÃ FIX: Chuyển torch.cuda.amp.GradScaler thành torch.amp.GradScaler để loại bỏ cảnh báo Deprecation
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

for exp_idx, cfg in enumerate(CONFIG_LIST):
    gc.collect()
    torch.cuda.empty_cache()

    exp_name = cfg['name']
    print(f'\n================ CHAY SANG LOC [{exp_idx+1}/{len(CONFIG_LIST)}]: {exp_name} ================')

    train_loader = DataLoader(train_sub_ds, batch_size=cfg.get('batch_size', 2), shuffle=True, num_workers=2, pin_memory=True)

    model = FORGEDeblurNet(
        img_channel=3,
        out_channels=3,
        width=COMMON_CONFIG['width'],
        enc_blk_nums=COMMON_CONFIG['enc_blk_nums'],
        middle_blk_num=COMMON_CONFIG['middle_blk_num'],
        dec_blk_nums=COMMON_CONFIG['dec_blk_nums'],
        num_queries=COMMON_CONFIG['num_queries'],
        num_experts=COMMON_CONFIG['num_experts'],
        r=COMMON_CONFIG['r'],
        rb=COMMON_CONFIG['rb'],
    ).to(device)

    model.load_pretrained(nafnet_state_dict)
    if cfg['freeze_backbone']:
        model.freeze_backbone(True)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'--> Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)')

    optimizer = optim.AdamW(trainable_params, lr=cfg['lr'], weight_decay=1e-4)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_epochs=cfg.get('warmup_epochs', 1),
        total_epochs=SCREENING_EPOCHS,
        min_lr=1e-7
    )

    best_val_psnr = 0.0
    best_val_ssim = 0.0

    start_time = time.time()
    history = {'train_loss': [], 'val_psnr': [], 'val_ssim': []}
    ckpt_path = os.path.join('weights', f'screening_{exp_name}.pth')

    for epoch in range(1, SCREENING_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f'[{exp_name}] Epoch {epoch:02d}/{SCREENING_EPOCHS:02d}')

        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    preds, aux_loss = model(inputs)
                    recon_loss = criterion(preds, targets)
                    total_loss = recon_loss + COMMON_CONFIG['aux_weight'] * aux_loss
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                preds, aux_loss = model(inputs)
                recon_loss = criterion(preds, targets)
                total_loss = recon_loss + COMMON_CONFIG['aux_weight'] * aux_loss
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                optimizer.step()

            running_loss += total_loss.item()
            pbar.set_postfix({'loss': f"{total_loss.item():.4f}", 'lr': f"{optimizer.param_groups[0]['lr']:.2e}"})

        epoch_loss = running_loss / len(train_loader)
        scheduler.step()

        val_psnr, val_ssim = evaluate_model(model, val_loader, desc='Validation')
        print(f"--> [{exp_name} | Epoch {epoch:02d}] Loss: {epoch_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | LR: {optimizer.param_groups[0]['lr']:.2e}")

        history['train_loss'].append(epoch_loss)
        history['val_psnr'].append(val_psnr)
        history['val_ssim'].append(val_ssim)

        if val_psnr > best_val_psnr:
            best_val_psnr = val_psnr
            best_val_ssim = val_ssim
            torch.save(model.state_dict(), ckpt_path)

    time_mins = (time.time() - start_time) / 60.0

    SCREENING_RESULTS[exp_name] = {
        'config': cfg,
        'best_val_psnr': best_val_psnr,
        'best_val_ssim': best_val_ssim,
        'history': history,
        'checkpoint': ckpt_path,
        'time_mins': time_mins
    }

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

best_exp_name = max(SCREENING_RESULTS, key=lambda k: SCREENING_RESULTS[k]['best_val_psnr'])
BEST_CONFIG = SCREENING_RESULTS[best_exp_name]['config']
print(f"\n--> Cau hinh tot nhat: [{best_exp_name}] (Val PSNR: {SCREENING_RESULTS[best_exp_name]['best_val_psnr']:.2f} dB)")


================ CHAY SANG LOC [1/3]: Exp1_MoE_LR_3e4 ================
--> Đã nạp thành công Checkpoint và cấy ghép SVD-Anchor FORGE MoE (Middle + Decoder).
--> Trainable: 5,808,303 / 68,450,543 (8.5%)


[Exp1_MoE_LR_3e4] Epoch 01/05:   0%|          | 0/852 [00:10<?, ?it/s]

[Validation]:   0%|          | 0/200 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 900.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 2.55 GiB is free. Process 12776 has 17179869184.00 GiB memory in use. Process 10624 has 17179869184.00 GiB memory in use. Process 17608 has 17179869184.00 GiB memory in use. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 4.20 GiB is allocated by PyTorch, and 107.41 MiB is reserved by PyTorch but unallocated.

### Bảng Xếp Hạng & Biểu Đồ So Sánh

In [ ]:
summary_data = []
for name, res in SCREENING_RESULTS.items():
    cfg = res['config']
    summary_data.append({
        'Tên Thí Nghiệm': name,
        'LR': cfg['lr'],
        'Batch Size': cfg['batch_size'],
        'Freeze': cfg['freeze_backbone'],
        'Best Val PSNR (dB)': round(res['best_val_psnr'], 2),
        'Best Val SSIM': round(res['best_val_ssim'], 4),
        'Mô Tả': cfg['description']
    })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(by='Best Val PSNR (dB)', ascending=False).reset_index(drop=True)
df_summary.index += 1
df_summary.index.name = 'Hạng'

print(" BẢNG SO SÁNH HIỆU NĂNG CÁC CẤU HÌNH SÀNG LỌC:")
display(df_summary)

best_exp_name = df_summary.iloc[0]['Tên Thí Nghiệm']
print(f"\n CẤU HÌNH TỐT NHẤT LÀ: [{best_exp_name}] với PSNR = {df_summary.iloc[0]['Best Val PSNR (dB)']} dB!")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['b', 'g', 'r', 'c', 'm', 'y']

for i, (name, res) in enumerate(SCREENING_RESULTS.items()):
    c = colors[i % len(colors)]
    hist = res['history']
    epochs = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(epochs, hist['train_loss'], label=name, color=c, marker='o')
    axes[1].plot(epochs, hist['val_psnr'], label=name, color=c, marker='s')

axes[0].set_title('Training Loss', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].set_title('Validation PSNR', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PSNR (dB)')
axes[1].legend()

plt.tight_layout()
plt.show()

### Huấn Luyện Tinh Chỉnh (Fine-Tuning) & Đánh Giá Sau Cùng

In [ ]:
print(f"\n===================================================")
print(f" BẮT ĐẦU HUẤN LUYỆN KỸ CẤU HÌNH: [{BEST_CONFIG['name']}]")
print(f"===================================================")

MAX_FINE_TUNE_EPOCHS = 15  
PATIENCE = 5               
MIN_DELTA = 0.02           

train_loader_full = DataLoader(train_sub_ds, batch_size=BEST_CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)

final_model = FORGEDeblurNet(
    img_channel=3, 
    out_channels=3,
    width=COMMON_CONFIG['width'],
    enc_blk_nums=COMMON_CONFIG['enc_blk_nums'],
    middle_blk_num=COMMON_CONFIG['middle_blk_num'],
    dec_blk_nums=COMMON_CONFIG['dec_blk_nums'],
    num_queries=COMMON_CONFIG['num_queries'], 
    num_experts=COMMON_CONFIG['num_experts'],
    r=COMMON_CONFIG['r'],
    rb=COMMON_CONFIG['rb']
).to(device)

best_screening_ckpt = SCREENING_RESULTS[BEST_CONFIG['name']]['checkpoint']
final_model.load_state_dict(torch.load(best_screening_ckpt, map_location=device))
print(f"--> Đã nạp thành công checkpoint từ vòng sàng lọc: {best_screening_ckpt}")

if BEST_CONFIG['freeze_backbone']:
    final_model.freeze_backbone(True)

trainable_params = [p for p in final_model.parameters() if p.requires_grad]

fine_tune_lr = min(BEST_CONFIG['lr'], 2e-4)
optimizer = optim.AdamW(trainable_params, lr=fine_tune_lr, weight_decay=1e-4)
scheduler = WarmupCosineScheduler(optimizer, warmup_epochs=1, total_epochs=MAX_FINE_TUNE_EPOCHS, min_lr=1e-7)

best_val_psnr = SCREENING_RESULTS[BEST_CONFIG['name']]['best_val_psnr']
no_improve_epochs = 0
best_checkpoint_path = os.path.join("weights", f"best_model_{BEST_CONFIG['name']}.pth")

import shutil
shutil.copyfile(best_screening_ckpt, best_checkpoint_path)

scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

for epoch in range(1, MAX_FINE_TUNE_EPOCHS + 1):
    final_model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader_full, desc=f"[Fine-Tuning] Epoch {epoch:02d}/{MAX_FINE_TUNE_EPOCHS:02d}")
    
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                preds, aux_loss = final_model(inputs)
                recon_loss = criterion(preds, targets)
                total_loss = recon_loss + COMMON_CONFIG['aux_weight'] * aux_loss
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            preds, aux_loss = final_model(inputs)
            recon_loss = criterion(preds, targets)
            total_loss = recon_loss + COMMON_CONFIG['aux_weight'] * aux_loss
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            
        running_loss += total_loss.item()
        pbar.set_postfix({'loss': f"{total_loss.item():.4f}", 'lr': f"{optimizer.param_groups[0]['lr']:.2e}"})
        
    epoch_loss = running_loss / len(train_loader_full)
    scheduler.step()
    
    val_psnr, val_ssim = evaluate_model(final_model, val_loader, desc="Validation Check")
    print(f"--> [Fine-Tuning Epoch {epoch:02d}] Train Loss: {epoch_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f}")
    
    if val_psnr - best_val_psnr > MIN_DELTA:
        best_val_psnr = val_psnr
        no_improve_epochs = 0
        torch.save(final_model.state_dict(), best_checkpoint_path)
        print(f"    [+] Độ chính xác cải thiện. Đã lưu checkpoint tại: {best_checkpoint_path}")
    else:
        no_improve_epochs += 1
        print(f"    [-] Độ chính xác không cải thiện ({no_improve_epochs}/{PATIENCE} epoch).")
        
    if no_improve_epochs >= PATIENCE:
        print(f"\n KÍCH HOẠT EARLY STOPPING. Dừng huấn luyện!")
        break

print("\n===================================================")
print(" ĐANG NẠP BEST CHECKPOINT & ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP")
print("===================================================")

if os.path.exists(best_checkpoint_path):
    final_model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
    print(f"--> Nạp thành công trọng số tối ưu từ: {best_checkpoint_path}")

final_test_psnr, final_test_ssim = evaluate_model(final_model, test_loader, desc="Final Test")

print("\n" + "="*55)
print("             KẾT QUẢ ĐÁNH GIÁ SAU CÙNG (FINAL TEST)")
print("="*55)
print(f" Mô hình baseline (NAFNet)   : PSNR = {baseline_psnr:.2f} dB | SSIM = {baseline_ssim:.4f}")
print(f" Mô hình FORGE ({BEST_CONFIG['name']})   : PSNR = {final_test_psnr:.2f} dB | SSIM = {final_test_ssim:.4f}")
diff_psnr = final_test_psnr - baseline_psnr
print(f" Mức độ cải thiện            : {diff_psnr:+.2f} dB PSNR")
print("="*55)


### Trực Quan Hóa Kết Quả (Visual Demo)

In [ ]:
def visualize_results(model, loader, num_imgs=3):
    model.eval()
    inputs, targets = next(iter(loader))
    inputs, targets = inputs.to(device), targets.to(device)
    with torch.no_grad():
        preds, _ = model(inputs)
        preds = torch.clamp(preds, 0.0, 1.0)
    
    inputs = inputs.cpu()
    targets = targets.cpu()
    preds = preds.cpu()
    
    batch_size = min(num_imgs, inputs.size(0))
    fig, axes = plt.subplots(batch_size, 3, figsize=(15, 5 * batch_size))
    
    if batch_size == 1:
        axes = [axes]
        
    for i in range(batch_size):
        # Hiển thị ảnh bị mờ (Input)
        axes[i][0].imshow(inputs[i].permute(1, 2, 0))
        axes[i][0].set_title("Input (Blur)")
        axes[i][0].axis('off')
        
        # Hiển thị ảnh sau khi làm nét (Prediction)
        axes[i][1].imshow(preds[i].permute(1, 2, 0))
        axes[i][1].set_title("Prediction (Deblurred)")
        axes[i][1].axis('off')
        
        # Hiển thị ảnh rõ nét gốc (Ground Truth)
        axes[i][2].imshow(targets[i].permute(1, 2, 0))
        axes[i][2].set_title("Ground Truth (Sharp)")
        axes[i][2].axis('off')
        
    plt.tight_layout()
    plt.show()

visualize_results(final_model, test_loader, num_imgs=2)